In [ ]:
import sisl
import matplotlib.pyplot as plt
import numpy as np
from TimedependentTransport import TD_Transport, AdaptiveRK4,hbar
from numba import njit
from funcs import diff_central, calc_current
from ase import Atoms
from ase.build import molecule
from ase.visualize import view


In [ ]:
#Define sampling of the lead Gammas
eps  = (1e-6 + 1j  * 1e-2)
line = np.linspace(-5, 5, 452) + eps
# line = np.array([-8, -4, -2.5, -1.0,  1.0, 2.0 ,3.0, 4.0, 5.0, 6.0, 7.0, 8.0]) + eps
line = np.vstack((line,line))
# make a molecule
BDA = molecule('BDA')
BDA.cell = np.diag(1.44 * np.ones(3))

#BDA.edit() COMMENT THIS IN IF YOU WANT TO MODIFY THE DEVICE WITH ASE

BDA = sisl.Geometry.fromASE(BDA)
# make leads
move_up = 5.0
move_dw = -14.8
L  = sisl.geom.sc(1.44, 'C').tile(8,1)
LU = L.move([0,move_up,0]).add_vacuum(10,0).add_vacuum(10,2)
LD = L.move([0,move_dw,0]).add_vacuum(10,0).add_vacuum(10,2)
dev = BDA.add(LU).add(LD)
dev = dev.add_vacuum(20,0).add_vacuum(dev.xyz[:,1].max() - dev.xyz[:,1].min(),1).add_vacuum(20,2)
dev = dev.move(dev.cell[1,:]/2)
LU  = sisl.geom.sc(1.44, 'C').tile(4,1).move([0,move_up,0]).add_vacuum(10,0).add_vacuum(10,2)
LD  = sisl.geom.sc(1.44, 'C').tile(4,1).move([0,move_dw,0]).add_vacuum(10,0).add_vacuum(10,2)
LU  = LU.move(LU.cell[1,:])
LD  = LD.move(-LD.cell[1,:]/2 + np.array([0, 2 * 1.44, 0]))
LU  = LU.move (dev.cell[1,:]/2)
LD  = LD.move (dev.cell[1,:]/2)

plt.show()
sisl.plot(dev); plt.axis('equal')

In [ ]:
Test = TD_Transport([LD,LU], dev, kT_i = [0.025, 0.025])
Test.Make_Contour(line, 20, pole_mode = 'JieHu2011')
plt.show()
Test.Electrodes( semi_infs = ['-a2', '+a2'] , kp=[[1,50,1],[1,50,1]])
Test.make_device()
plt.show()
Test.Device.Visualise()
plt.show()

In [ ]:
Test.run_electrodes()
Test.run_device()

In [ ]:
# Get the pz part of the system. Hydrogens should only affect the Carbon onsite potential

sub_indices = []
TBT = sisl.get_sile(Test.Device.dir + '/siesta.TBT.nc')
count = 0
atoms = [TBT.geom.atoms[i] for i in range(TBT.geom.na) if i in TBT.a_dev]
for ia, a in enumerate(atoms):
    for io in range(TBT.geom.orbitals[ia]):
        if io == 2:
            sub_indices += [count]
        count+=1

Test.read_data()
Test.Fit(fact = 2.0, Fallback_W = 30.0, NumL = 25, fit_mode = 'all_from_SE', 
              use_analytical_jac = True,  min_method = 'L-BFGS-B',
              ebounds = (line.real.min()-1, line.real.max()+1), 
              wbounds = (0.001, 20.0), 
              gbounds = (None, None),
              tol = -1.0, options = {'disp':True,'maxiter':200, 'gtol':1e-5, 'iprint':1}, 
              fit_real_part = False,
              specific_bounds = None, 
              alpha_PO = 0.0)

plt.show()
Test.diagonalise()
Test.get_propagation_quantities(); 
Test.get_dense_matrices()

Vi = np.linspace(-.4,.4,20); idx = np.argsort(np.abs(Vi)); Vi = Vi[idx]
Test.run_device_non_eq(Vi)
h,v = Test.neq_Hs()

In [ ]:
I,J,i,j  = 0,0,1,1 #I,J Blocks from tbtrans, i,j are the subindices in these blocks
Test.Inspect_Lorentzian_fit(1,I,J,i,j, Emin = -5,Emax= 5)

In [ ]:
Test.Inspect_Transmission(0,1)

In [ ]:
Test.get_propagation_quantities()
Test.get_dense_matrices()
sig = Test.sigma
psi = Test.Psi_vec
omega = Test.omegas
no_d = sig.shape[2]

In [ ]:
# Spline interpolation:
from scipy.interpolate import CubicSpline
cs = CubicSpline(v, h, axis = 0)
k = 3
x,c = cs.x, cs.c

@njit
def H_spline(V):
    dV = V - v
    dV[dV<0] = 10**5
    i = np.where(dV == dV.min())[0][0]
    res = np.zeros((1,no_d, no_d),dtype = np.complex128)
    for m in range(k+1):
        res += c[m, i] * (V - x[i])**(k-m)
    return res
    



In [ ]:
from Peter_Uhd import air_photonics_pulse

V = 1.0
@njit
def Bias(t, tp, ts):
    return V * (np.tanh(t / ts) - np.tanh((t - tp) / ts)) / 2
@njit
def fd(x, mu, s):
    return 1 / (1 + np.exp((x - mu) / s))
@njit
def delta(t, a):
    if a == 0:
        return air_photonics_pulse(t)  #Bias(t, 20, 1)/2
    elif a == 1:
        return -air_photonics_pulse(t)#- Bias(t, 20, 1)/2
@njit
def zero_bias(t, a):
    return 0.0
@njit
def zero_dH(t, sigma):
    A = np.zeros((1, no_d, no_d), dtype=np.complex128)
    return A

H0 = Test.Hdense[:,0,:,:].copy()

@njit
def dH(t, sigma):
    # We can put in Hartree correction here like in 
    # "Insights into the Charge Separation Dynamics in Photoexcited
    # Molecular Junctions"
    return H0 - H_spline(delta(t,0) - detal(t,1))

f   = Test.make_f_general(parallel = True, fastmath = True)
xi  = Test.xi
Ixi = Test.Ixi

t1, data1 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-5, -15.0, 10.0,
                        zero_dH,
                        zero_bias,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
 )

sig          = data1['last sigma']
omega        = data1['last omega']
psi          = data1['last psi']

t2, data2 = AdaptiveRK4(
                        f,
                        sig, psi, omega,
                        1e-6, -5.0, 500.0,
                        dH,
                        delta,
                        Test.Ixi,
                        0.01,
                        fixed_mode=False,
                        name="Multithread_RK4",
)